# Soundscape Generator — AI Backend

AudioGen (medium) + MusicGen (large) on Colab GPU, exposed as a FastAPI endpoint via ngrok.

**Setup:** Runtime → Change runtime type → T4 GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/orcrobert/soundscape-generator/blob/main/colab/soundscape_generator.ipynb)

In [ ]:
!pip install av==12.0.0 -q
!pip install --no-deps git+https://github.com/facebookresearch/audiocraft.git -q
!pip install encodec lameenc num2words spacy demucs cloudpickle flashy torchmetrics torchdiffeq pyngrok fastapi uvicorn xformers -q

In [ ]:
NGROK_TOKEN = "YOUR_NGROK_TOKEN"
AUDIOGEN_MODEL = "facebook/audiogen-medium"
MUSICGEN_MODEL = "facebook/musicgen-large"
DURATION = 10

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import torch, base64, threading, uvicorn
from audiocraft.models import AudioGen, MusicGen
from audiocraft.data.audio import audio_write
from pyngrok import ngrok

app = FastAPI()

print(f"Loading {AUDIOGEN_MODEL}...")
audiogen = AudioGen.get_pretrained(AUDIOGEN_MODEL, device="cuda")
audiogen.set_generation_params(duration=DURATION)

print(f"Loading {MUSICGEN_MODEL}...")
musicgen = MusicGen.get_pretrained(MUSICGEN_MODEL, device="cuda")
musicgen.set_generation_params(duration=DURATION)

print(f"Models ready. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


class GenerateRequest(BaseModel):
    elements: list[dict]


def _generate_batch(model, elements):
    wavs = model.generate([e["prompt"] for e in elements])
    results = []
    for elem, wav in zip(elements, wavs):
        path = f"/tmp/{elem['label'].replace(' ', '_')}.wav"
        audio_write(path, wav.cpu(), model.sample_rate, strategy="loudness", loudness_compressor=True)
        with open(path + ".wav", "rb") as f:
            elem["audio_b64"] = base64.b64encode(f.read()).decode()
        results.append(elem)
    return results


@app.post("/generate")
def generate(req: GenerateRequest):
    sounds = [e for e in req.elements if e["type"] == "sound"]
    music = [e for e in req.elements if e["type"] == "music"]
    results = []
    if sounds:
        results.extend(_generate_batch(audiogen, sounds))
    if music:
        results.extend(_generate_batch(musicgen, music))
    return {"elements": results}


ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000)
print(f"\nAPI ready at: {public_url}")
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000)).start()